# Inverse Correspondence: Regular vs. Inverted Logistic Map

**Question**: Is there a useful correspondence between the dynamics of a map and its alpha-inverted counterpart ($\alpha = -1$)?

The "naive inverse" $g(x) = -f(x) + 2x$ flips the deviation from the bisector: where $f$ pushes away from the identity, $g$ pushes inward, and vice versa. Fixed points are preserved but their stability is inverted:

$$g'(x^*) = 2 - f'(x^*)$$

So if $f'(x^*) = -1$ (period-2 boundary), then $g'(x^*) = 3$ — violently unstable. And if $f'(x^*) = 3$ (period-3 window boundary), then $g'(x^*) = -1$ — the *inverted* system's period-doubling onset.

**This section is exploratory.** We look at both systems across the full parameter range and ask what, if anything, the inverted system reveals about the regular one.

**Roadmap:**

| § | Section | What we're looking for |
|---|---------|----------------------|
| 1 | Setup | Dual cobweb + bifurcation infrastructure |
| 2 | Dual bifurcation | Do the bifurcation trees interlock? |
| 3 | Period analysis | Periodicity of regular vs. inverted at key $a$ values |
| 4 | Lyapunov exponents | Do chaos measures anti-correlate? |
| 5 | Intermediate $\alpha$ | What happens between regular and inverse? |
| 6 | $\alpha(x)$ experiments | Does a position-dependent $\alpha$ couple the two behaviors? |
| 7 | Observations | What patterns (if any) emerge? |

In [ ]:
import sys, importlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

sys.path.insert(0, '..')
import petrification.maps as maps
import petrification.iteration as iteration
import petrification.bifurcation as bif
import petrification.transforms as transforms
for mod in [maps, iteration, bif, transforms]:
    importlib.reload(mod)

from petrification.maps import logistic
from petrification.iteration import iterate, iterate_transformed, cobweb_data, find_fixed_points
from petrification.bifurcation import compute_bifurcation, compute_bifurcation_transformed

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
np.set_printoptions(precision=8)

## §1. Setup: The Regular and Inverted Maps

For the logistic map $f(x) = ax(1-x)$, the $\alpha$-transform is:

$$g_\alpha(x) = \alpha\,f(x) + (1-\alpha)\,x$$

At $\alpha = 1$: regular map. At $\alpha = -1$: the "naive inverse":

$$g_{-1}(x) = -f(x) + 2x = -ax(1-x) + 2x = x(2 + a - ax) - ax^2$$

**Stability inversion**: At a fixed point $x^*$ where $f(x^*) = x^*$:

$$g_\alpha'(x^*) = 1 + \alpha(f'(x^*) - 1) = 1 + \alpha(\lambda - 1)$$

For $\alpha = -1$: $g'(x^*) = 2 - \lambda$. The stability is reflected through $\lambda = 1$:
- $\lambda < 1$ (stable under $f$) $\Rightarrow$ $g' > 1$ (unstable under $g$)
- $\lambda > 1$ (unstable under $f$) $\Rightarrow$ $g' < 1$ (stable under $g$, unless $g' < -1$)

In [ ]:
# --- Dual cobweb comparison at four key a values ---
a_values = [2.8, 3.2, 3.83, 4.0]
labels   = ['Stable FP\n(a=2.8)', 'Period 2\n(a=3.2)', 'Period 3 window\n(a=3.83)', 'Chaos\n(a=4.0)']
x0 = 0.4

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, (a, label) in enumerate(zip(a_values, labels)):
    lx = np.linspace(-0.1, 1.1, 500)
    
    # --- Regular map (top row) ---
    ax = axes[0, col]
    Ix, Iy = cobweb_data(logistic, a, x0, n_iter=40)
    ax.plot(lx, logistic(a, lx), 'b-', lw=2)
    ax.plot(lx, lx, 'k-', lw=0.8)
    ax.plot(Ix, Iy, 'r-', lw=0.6, alpha=0.7)
    ax.set_title(f'Regular (α=1)\n{label}')
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
    if col == 0: ax.set_ylabel(r'$x_{n+1}$')
    
    # Fixed points and slopes
    fps = find_fixed_points(logistic, a, (0, 1))
    for fp in fps:
        lam = a - 2*a*fp
        ax.plot(fp, fp, 'go' if abs(lam) < 1 else 'ro', ms=8, zorder=5)
    
    # --- Inverted map (bottom row) ---
    ax = axes[1, col]
    Ix_inv, Iy_inv = cobweb_data(logistic, a, x0, n_iter=40, alpha=-1)
    g_inv = -1 * logistic(a, lx) + 2 * lx
    ax.plot(lx, g_inv, 'purple', lw=2)
    ax.plot(lx, lx, 'k-', lw=0.8)
    ax.plot(Ix_inv, Iy_inv, 'orange', lw=0.6, alpha=0.7)
    ax.set_title(f'Inverted (α=−1)\n{label}')
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel(r'$x_n$')
    if col == 0: ax.set_ylabel(r'$x_{n+1}$')
    
    # Same fixed points, inverted stability
    for fp in fps:
        lam = a - 2*a*fp
        g_prime = 2 - lam
        ax.plot(fp, fp, 'go' if abs(g_prime) < 1 else 'ro', ms=8, zorder=5)

plt.suptitle('Regular vs. Inverted Logistic Map: Cobweb Diagrams', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Print stability table
print(f"{'a':>5}  {'x*':>8}  {'df/dx':>8}  {'dg/dx':>8}  {'f stable':>10}  {'g stable':>10}")
print("-" * 62)
for a in a_values:
    fps = find_fixed_points(logistic, a, (0, 1))
    for fp in fps:
        lam = a - 2*a*fp
        g_p = 2 - lam
        print(f"{a:5.2f}  {fp:8.4f}  {lam:+8.4f}  {g_p:+8.4f}  {'✓' if abs(lam)<1 else '✗':>10}  {'✓' if abs(g_p)<1 else '✗':>10}")

## §2. Dual Bifurcation Diagrams

The regular and inverted bifurcation diagrams, overlaid. Do the bifurcation trees interlock? Where one has a period-$n$ window, does the other show something complementary?

In [ ]:
# --- Dual bifurcation diagram ---
a_range = np.linspace(2.5, 4.0, 600)
x0_samples = np.linspace(0.1, 0.9, 20)

# Regular (α=1)
a_reg, x_reg = compute_bifurcation(logistic, a_range, x0_samples,
                                    n_iter=600, n_discard=400)
# Inverted (α=-1)
a_inv, x_inv = compute_bifurcation_transformed(logistic, -1.0, a_range, x0_samples,
                                                n_iter=600, n_discard=400, x_max=10.0)

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

# Panel 1: Regular
axes[0].scatter(a_reg, x_reg, s=0.1, c='blue', alpha=0.4)
axes[0].set_ylabel('$x$ (regular)')
axes[0].set_title(r'Regular logistic map ($\alpha = 1$)')
axes[0].set_ylim(-0.1, 1.1)
# Mark key transitions
for av, lab in [(3.0, 'P-2'), (3.449, 'P-4'), (3.5699, 'Chaos'), (3.83, 'P-3')]:
    axes[0].axvline(av, color='gray', ls='--', alpha=0.4, lw=0.8)
    axes[0].text(av, 1.05, lab, ha='center', fontsize=8, color='gray')

# Panel 2: Inverted
axes[1].scatter(a_inv, x_inv, s=0.1, c='purple', alpha=0.4)
axes[1].set_ylabel('$x$ (inverted)')
axes[1].set_title(r'Inverted logistic map ($\alpha = -1$)')
axes[1].set_ylim(-0.5, 1.5)
for av in [3.0, 3.449, 3.5699, 3.83]:
    axes[1].axvline(av, color='gray', ls='--', alpha=0.4, lw=0.8)

# Panel 3: Overlaid
axes[2].scatter(a_reg, x_reg, s=0.1, c='blue', alpha=0.3, label='Regular')
axes[2].scatter(a_inv, x_inv, s=0.1, c='red', alpha=0.3, label='Inverted')
axes[2].set_xlabel('Parameter $a$')
axes[2].set_ylabel('$x$')
axes[2].set_title('Overlaid: do the bifurcation trees interlock?')
axes[2].set_ylim(-0.5, 1.5)
axes[2].legend(loc='upper left', markerscale=20)
for av in [3.0, 3.449, 3.5699, 3.83]:
    axes[2].axvline(av, color='gray', ls='--', alpha=0.4, lw=0.8)

plt.tight_layout()
plt.show()

## §3. Period Analysis: Regular vs. Inverted at Each $a$

For each $a$, we detect the period of the attractor for both the regular and inverted maps. If there's a correspondence, it should show up here — e.g., do period-$n$ windows in the regular map correspond to period-$n$ (or period-$m$) windows in the inverted map?

In [ ]:
def detect_period(f, a, x0=0.4, alpha=None, n_warmup=2000, n_test=200, tol=1e-8):
    """Detect the period of the attractor by looking for repeats after warmup."""
    x = x0
    for _ in range(n_warmup):
        if alpha is not None:
            a_val = alpha(x) if callable(alpha) else alpha
            x = a_val * f(a, x) + (1 - a_val) * x
        else:
            x = f(a, x)
        if not np.isfinite(x) or abs(x) > 1e6:
            return -1  # diverged
    
    # Record attractor points
    attractor = [x]
    for _ in range(n_test):
        if alpha is not None:
            a_val = alpha(x) if callable(alpha) else alpha
            x = a_val * f(a, x) + (1 - a_val) * x
        else:
            x = f(a, x)
        if not np.isfinite(x) or abs(x) > 1e6:
            return -1
        attractor.append(x)
    
    attractor = np.array(attractor)
    
    # Check for period by looking at x_{n+p} == x_n
    for p in range(1, min(65, n_test)):
        if np.all(np.abs(attractor[p:] - attractor[:-p]) < tol):
            return p
    
    return 0  # aperiodic (chaos)


# --- Scan both maps across the full parameter range ---
a_scan = np.linspace(2.5, 4.0, 800)
period_reg = np.array([detect_period(logistic, a) for a in a_scan])
period_inv = np.array([detect_period(logistic, a, alpha=-1.0) for a in a_scan])

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Regular period
axes[0].scatter(a_scan, period_reg, s=2, c='blue', alpha=0.6)
axes[0].set_ylabel('Period (regular)')
axes[0].set_title(r'Detected period: regular map ($\alpha=1$)')
axes[0].set_ylim(-1.5, 35)
axes[0].axhline(0, color='red', ls=':', alpha=0.5, label='Chaos (period=0)')
axes[0].legend()

# Panel 2: Inverted period
axes[1].scatter(a_scan, period_inv, s=2, c='purple', alpha=0.6)
axes[1].set_ylabel('Period (inverted)')
axes[1].set_title(r'Detected period: inverted map ($\alpha=-1$)')
axes[1].set_ylim(-1.5, 35)
axes[1].axhline(0, color='red', ls=':', alpha=0.5)
axes[1].axhline(-1, color='gray', ls=':', alpha=0.3, label='Diverged')
axes[1].legend()

# Panel 3: Overlaid
c_reg = np.where(period_reg == 0, 'red', np.where(period_reg == -1, 'gray', 'blue'))
c_inv = np.where(period_inv == 0, 'red', np.where(period_inv == -1, 'gray', 'purple'))
axes[2].scatter(a_scan, period_reg, s=3, c='blue', alpha=0.5, label='Regular')
axes[2].scatter(a_scan, -period_inv, s=3, c='purple', alpha=0.5, label='Inverted (negated)')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_xlabel('Parameter $a$')
axes[2].set_ylabel('Period')
axes[2].set_title('Overlaid (inverted period negated for visual separation)')
axes[2].legend()
axes[2].set_ylim(-35, 35)

for ax in axes:
    for av in [3.0, 3.449, 3.5699, 3.83]:
        ax.axvline(av, color='gray', ls='--', alpha=0.3, lw=0.8)

plt.tight_layout()
plt.show()

# Print a table at key a values
print(f"\n{'a':>6}  {'Period (reg)':>12}  {'Period (inv)':>12}  ")
print("-" * 36)
key_a = [2.8, 3.0, 3.2, 3.449, 3.5, 3.5699, 3.6, 3.83, 3.84, 4.0]
for a in key_a:
    p_r = detect_period(logistic, a)
    p_i = detect_period(logistic, a, alpha=-1.0)
    p_r_str = 'chaos' if p_r == 0 else ('div' if p_r == -1 else str(p_r))
    p_i_str = 'chaos' if p_i == 0 else ('div' if p_i == -1 else str(p_i))
    print(f"{a:6.4f}  {p_r_str:>12}  {p_i_str:>12}")

## §4. Lyapunov Exponents: Regular vs. Inverted

The Lyapunov exponent measures the rate of exponential divergence of nearby orbits. For a map iterated $N$ times:

$$\Lambda = \frac{1}{N}\sum_{i=0}^{N-1} \ln|g'(x_i)|$$

For the alpha-transform $g_\alpha(x) = \alpha f(x) + (1-\alpha)x$:

$$g_\alpha'(x) = \alpha f'(x) + (1-\alpha)$$

At $\alpha = -1$: $g'(x) = -f'(x) + 2$. So the Lyapunov exponents are related but NOT simply negated. Do they anti-correlate, or is there a subtler relationship?

In [ ]:
def lyapunov(f, a, x0=0.4, alpha=None, n_warmup=500, n_iter=5000):
    """Compute the Lyapunov exponent along a trajectory."""
    x = x0
    # Warmup
    for _ in range(n_warmup):
        if alpha is not None:
            a_val = alpha(x) if callable(alpha) else alpha
            x = a_val * f(a, x) + (1 - a_val) * x
        else:
            x = f(a, x)
        if not np.isfinite(x) or abs(x) > 1e6:
            return np.nan
    
    log_sum = 0.0
    for _ in range(n_iter):
        # Derivative of logistic: f'(x) = a(1-2x)
        fp = a * (1 - 2*x)
        if alpha is not None:
            a_val = alpha(x) if callable(alpha) else alpha
            gp = a_val * fp + (1 - a_val)
        else:
            gp = fp
        
        if abs(gp) < 1e-15:
            log_sum += -30  # very stable
        else:
            log_sum += np.log(abs(gp))
        
        if alpha is not None:
            a_val = alpha(x) if callable(alpha) else alpha
            x = a_val * f(a, x) + (1 - a_val) * x
        else:
            x = f(a, x)
        
        if not np.isfinite(x) or abs(x) > 1e6:
            return np.nan
    
    return log_sum / n_iter


# --- Compute Lyapunov exponents across parameter range ---
a_lyap = np.linspace(2.5, 4.0, 800)
lam_reg = np.array([lyapunov(logistic, a) for a in a_lyap])
lam_inv = np.array([lyapunov(logistic, a, alpha=-1.0) for a in a_lyap])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Regular Lyapunov
axes[0, 0].plot(a_lyap, lam_reg, 'b-', lw=0.5, alpha=0.7)
axes[0, 0].axhline(0, color='red', ls='-', lw=1, alpha=0.5)
axes[0, 0].set_ylabel(r'$\Lambda$ (regular)')
axes[0, 0].set_title(r'Lyapunov exponent: regular ($\alpha=1$)')

# Panel 2: Inverted Lyapunov
v_inv = np.isfinite(lam_inv)
axes[0, 1].plot(a_lyap[v_inv], lam_inv[v_inv], 'purple', lw=0.5, alpha=0.7)
axes[0, 1].axhline(0, color='red', ls='-', lw=1, alpha=0.5)
axes[0, 1].set_ylabel(r'$\Lambda$ (inverted)')
axes[0, 1].set_title(r'Lyapunov exponent: inverted ($\alpha=-1$)')

# Panel 3: Overlaid
axes[1, 0].plot(a_lyap, lam_reg, 'b-', lw=0.5, alpha=0.6, label='Regular')
axes[1, 0].plot(a_lyap[v_inv], lam_inv[v_inv], 'purple', lw=0.5, alpha=0.6, label='Inverted')
axes[1, 0].axhline(0, color='red', ls='-', lw=0.8, alpha=0.5)
axes[1, 0].set_xlabel('$a$')
axes[1, 0].set_ylabel(r'$\Lambda$')
axes[1, 0].set_title('Overlaid Lyapunov exponents')
axes[1, 0].legend()

# Panel 4: Scatter — correlation?
both_valid = np.isfinite(lam_reg) & np.isfinite(lam_inv)
axes[1, 1].scatter(lam_reg[both_valid], lam_inv[both_valid], s=3, c=a_lyap[both_valid],
                   cmap='viridis', alpha=0.6)
axes[1, 1].set_xlabel(r'$\Lambda_\mathrm{regular}$')
axes[1, 1].set_ylabel(r'$\Lambda_\mathrm{inverted}$')
axes[1, 1].set_title(r'$\Lambda_\mathrm{inv}$ vs $\Lambda_\mathrm{reg}$ (color = $a$)')
axes[1, 1].axhline(0, color='gray', ls=':', alpha=0.5)
axes[1, 1].axvline(0, color='gray', ls=':', alpha=0.5)
# Add quadrant labels
axes[1, 1].text(0.02, 0.98, 'reg stable\ninv unstable', transform=axes[1, 1].transAxes,
               fontsize=8, va='top', ha='left', color='blue')
axes[1, 1].text(0.98, 0.02, 'reg chaotic\ninv stable', transform=axes[1, 1].transAxes,
               fontsize=8, va='bottom', ha='right', color='purple')
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('$a$')

for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
    for av in [3.0, 3.449, 3.5699, 3.83]:
        ax.axvline(av, color='gray', ls='--', alpha=0.3, lw=0.8)

plt.tight_layout()
plt.show()

# Correlation
if np.sum(both_valid) > 10:
    corr = np.corrcoef(lam_reg[both_valid], lam_inv[both_valid])[0, 1]
    print(f"Pearson correlation between Λ_reg and Λ_inv: {corr:.4f}")
    print(f"  (1 = perfectly correlated, -1 = anti-correlated, 0 = no correlation)")

## §5. Sweeping $\alpha$: What Happens Between Regular and Inverse?

We fix $a$ at interesting values and sweep $\alpha$ from $+1$ (regular) through $0$ (identity, no dynamics) to $-1$ (inverted). For each $(\alpha, a)$, we compute the Lyapunov exponent and period.

This shows the **transition** between the two systems and whether there are critical $\alpha$ values where the dynamics qualitatively change.

In [ ]:
# --- α sweep at key a values ---
alphas = np.linspace(-1.5, 1.5, 300)
a_test_vals = [3.2, 3.5, 3.83, 4.0]

fig, axes = plt.subplots(2, len(a_test_vals), figsize=(20, 10))

for col, a_t in enumerate(a_test_vals):
    # Lyapunov vs α
    lam_vs_alpha = np.array([lyapunov(logistic, a_t, alpha=al) for al in alphas])
    
    # Period vs α
    per_vs_alpha = np.array([detect_period(logistic, a_t, alpha=al) for al in alphas])
    
    # Top row: Lyapunov
    v = np.isfinite(lam_vs_alpha)
    axes[0, col].plot(alphas[v], lam_vs_alpha[v], 'k-', lw=1)
    axes[0, col].axhline(0, color='red', ls=':', alpha=0.5)
    axes[0, col].axvline(1, color='blue', ls='--', alpha=0.3, label=r'$\alpha=1$')
    axes[0, col].axvline(-1, color='purple', ls='--', alpha=0.3, label=r'$\alpha=-1$')
    axes[0, col].axvline(0, color='gray', ls=':', alpha=0.3)
    axes[0, col].set_title(f'$a = {a_t}$')
    if col == 0: axes[0, col].set_ylabel(r'$\Lambda$')
    axes[0, col].legend(fontsize=8)
    
    # Bottom row: Period
    valid_p = per_vs_alpha >= 0
    chaos_mask = per_vs_alpha == 0
    div_mask = per_vs_alpha == -1
    periodic_mask = per_vs_alpha > 0
    
    axes[1, col].scatter(alphas[periodic_mask], per_vs_alpha[periodic_mask],
                         s=5, c='blue', alpha=0.6, label='Periodic')
    axes[1, col].scatter(alphas[chaos_mask], np.zeros(np.sum(chaos_mask)),
                         s=5, c='red', alpha=0.6, label='Chaos')
    axes[1, col].scatter(alphas[div_mask], -np.ones(np.sum(div_mask)),
                         s=5, c='gray', alpha=0.3, label='Diverged')
    axes[1, col].axvline(1, color='blue', ls='--', alpha=0.3)
    axes[1, col].axvline(-1, color='purple', ls='--', alpha=0.3)
    axes[1, col].set_xlabel(r'$\alpha$')
    if col == 0: axes[1, col].set_ylabel('Period')
    axes[1, col].set_ylim(-2, 30)
    axes[1, col].legend(fontsize=7)

plt.suptitle(r'Lyapunov exponent and period vs. $\alpha$ at fixed $a$', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## §6. Position-Dependent $\alpha(x)$: Coupling Regular and Inverted

Now the interesting experiment. If $\alpha(x)$ varies — say $\alpha > 0$ in one region and $\alpha < 0$ in another — the map is regular in one part of phase space and inverted in the other. Does this create new dynamical structures?

We try several $\alpha(x)$ profiles:
1. **Step function**: $\alpha = +1$ for $x < 0.5$, $\alpha = -1$ for $x > 0.5$
2. **Linear ramp**: $\alpha(x) = 1 - 4x$ (crosses zero at $x = 0.25$, hits $-1$ at $x = 0.5$)
3. **Sinusoidal**: $\alpha(x) = \cos(\pi x)$ (smoothly oscillates between regular and inverted)

In [ ]:
# --- Position-dependent α(x) profiles ---
alpha_profiles = {
    'Step: +1 / −1': lambda x: 1.0 if x < 0.5 else -1.0,
    'Linear ramp':   lambda x: 1.0 - 4.0 * x,
    'Sinusoidal':    lambda x: np.cos(np.pi * x),
}

a_test = 3.83  # period-3 window — interesting dynamics

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (label, alpha_func) in enumerate(alpha_profiles.items()):
    # Top row: cobweb diagrams
    ax = axes[0, col]
    lx = np.linspace(-0.1, 1.1, 500)
    
    # Plot the alpha-modulated map
    g_vals = np.array([alpha_func(xi) * logistic(a_test, xi) + (1 - alpha_func(xi)) * xi for xi in lx])
    ax.plot(lx, g_vals, 'k-', lw=2)
    ax.plot(lx, lx, 'gray', lw=0.8)
    
    # Cobweb
    Ix, Iy = cobweb_data(logistic, a_test, 0.4, n_iter=80, alpha=alpha_func)
    ax.plot(Ix, Iy, 'r-', lw=0.5, alpha=0.7)
    
    ax.set_title(f'α(x) = {label}\n$a = {a_test}$')
    ax.set_xlim(-0.2, 1.2); ax.set_ylim(-0.5, 1.5)
    if col == 0: ax.set_ylabel(r'$x_{n+1}$')
    ax.set_xlabel(r'$x_n$')
    
    # Color the background by α sign
    ax.axvspan(-0.2, 1.2, alpha=0.05, color='blue')  # base
    # Show α profile as a secondary axis
    ax2 = ax.twinx()
    alpha_vals = np.array([alpha_func(xi) for xi in lx])
    ax2.plot(lx, alpha_vals, 'g-', lw=1, alpha=0.5)
    ax2.axhline(0, color='green', ls=':', alpha=0.3)
    ax2.set_ylabel(r'$\alpha(x)$', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    ax2.set_ylim(-3, 3)
    
    # Bottom row: trajectory time series
    ax = axes[1, col]
    traj = iterate_transformed(logistic, alpha_func, a_test, 0.4, n_iter=200)
    ax.plot(traj, 'k-', lw=0.5)
    ax.set_xlabel('Iteration $n$')
    if col == 0: ax.set_ylabel('$x_n$')
    ax.set_title(f'Trajectory')
    
    # Detect period
    p = detect_period(logistic, a_test, alpha=alpha_func)
    p_str = f'Period {p}' if p > 0 else ('Chaos' if p == 0 else 'Diverged')
    ax.text(0.95, 0.95, p_str, transform=ax.transAxes, fontsize=11, va='top', ha='right',
            bbox=dict(boxstyle='round', fc='wheat', alpha=0.8))

plt.suptitle(f'Position-dependent α(x) at $a = {a_test}$ (Period-3 window)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Compare with constant-α results
print(f"\nAt a = {a_test}:")
print(f"  Regular (α=+1): period = {detect_period(logistic, a_test)}")
print(f"  Inverted (α=−1): period = {detect_period(logistic, a_test, alpha=-1.0)}")
for label, af in alpha_profiles.items():
    p = detect_period(logistic, a_test, alpha=af)
    p_str = f'Period {p}' if p > 0 else ('Chaos' if p == 0 else 'Diverged')
    print(f"  α(x) = {label}: {p_str}")

### §5b. The Full $(\alpha, a)$ Phase Diagram

A 2D heatmap: for each $(a, \alpha)$, compute the Lyapunov exponent. This reveals the full landscape of regular/chaotic/divergent behavior across both parameters simultaneously.

In [ ]:
# --- Full (α, a) Lyapunov heatmap ---
a_heat = np.linspace(2.5, 4.0, 200)
alpha_heat = np.linspace(-1.5, 1.5, 200)
lyap_grid = np.full((len(alpha_heat), len(a_heat)), np.nan)

for i, al in enumerate(alpha_heat):
    for j, a_v in enumerate(a_heat):
        lyap_grid[i, j] = lyapunov(logistic, a_v, alpha=al, n_warmup=300, n_iter=2000)

fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Clip for visualization
lyap_clipped = np.clip(lyap_grid, -3, 3)
lyap_clipped[~np.isfinite(lyap_clipped)] = 3  # diverged → hot

im = ax.imshow(lyap_clipped, extent=[2.5, 4.0, -1.5, 1.5], aspect='auto',
               origin='lower', cmap='RdBu_r', vmin=-3, vmax=3)
ax.set_xlabel('Parameter $a$')
ax.set_ylabel(r'$\alpha$')
ax.set_title(r'Lyapunov exponent $\Lambda(\alpha, a)$: blue = stable, red = chaotic/diverged')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$\Lambda$')

# Mark key lines
ax.axhline(1, color='white', ls='--', lw=1, alpha=0.7, label=r'$\alpha=1$ (regular)')
ax.axhline(-1, color='cyan', ls='--', lw=1, alpha=0.7, label=r'$\alpha=-1$ (inverted)')
ax.axhline(0, color='yellow', ls=':', lw=1, alpha=0.5, label=r'$\alpha=0$ (identity)')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

## §7. Observations

*This section is intentionally left for observations after running the cells above. What patterns emerge?*

### Questions to answer from the data:

1. **Bifurcation interlocking** (§2): Do the regular and inverted bifurcation trees have complementary structure? Where one splits, does the other merge?

2. **Period correspondence** (§3): At $a = 3.83$ (period-3 window), what period does the inverted map show? Is there a simple rule like "period-$p$ regular $\Leftrightarrow$ period-$q$ inverted"?

3. **Lyapunov anti-correlation** (§4): Does $\Lambda_\text{reg} > 0$ (chaos) imply $\Lambda_\text{inv} < 0$ (stable), or is the relationship more complex?

4. **α sweep** (§5): Is there a critical $\alpha^*$ where the dynamics transition between regular-type and inverted-type behavior? Does $\alpha = 0$ (identity map) always separate them?

5. **Phase diagram** (§5b): What is the shape of the stable region in the $(\alpha, a)$ plane? Does the period-3 window visible at $\alpha = 1$ persist at other $\alpha$ values?

6. **Position-dependent coupling** (§6): Does mixing regular and inverted via $\alpha(x)$ create genuinely new dynamical structures (e.g., strange attractors not present at any constant $\alpha$)?

### The deeper question

If the dual system carries information about the regular system's dynamics, this suggests that the "$\alpha$-family" $\{g_\alpha\}_{\alpha \in \mathbb{R}}$ encodes the logistic map's dynamics more richly than the map alone. The full $(\alpha, a)$ phase diagram is the natural object to study — the regular map is just the $\alpha = 1$ slice.

### Connection to OGY Chaos Control

The §6 experiments with position-dependent $\alpha(x)$ are closely related to the **Ott–Grebogi–Yorke (OGY) method** of chaos control [Ott, Grebogi, Yorke, *Phys. Rev. Lett.* 64, 1196 (1990)].

**OGY in brief:** A chaotic attractor contains infinitely many unstable periodic orbits (UPOs). OGY stabilizes a chosen UPO by applying small, time-dependent parameter perturbations *only when the trajectory passes near the target orbit*. The perturbation is computed from the local linearization and applied as a feedback correction.

**The $\alpha(x)$ formulation as static OGY:** Our position-dependent $\alpha(x)$ achieves the same effect — targeted stabilization of an unstable fixed point — but *without* requiring real-time feedback or orbit detection. The key result is:

$$g'(x^*) = 1 + \alpha(x^*)\bigl(f'(x^*) - 1\bigr)$$

where the $\alpha'$ term vanishes at fixed points. This means:

1. **Localization**: A Gaussian bump $\alpha(x) = 1 - (1-\alpha_\text{ss})\exp(-(x-x^*)^2/2\sigma^2)$ modifies the dynamics only near the target, leaving the rest of the map essentially unchanged.
2. **Superstable control**: Setting $\alpha(x^*) = 1/(a-1)$ gives $g'(x^*) = 0$ — not merely stable, but **superstable** (error squares each iteration).
3. **Ergodic capture**: A chaotic orbit is ergodic — it visits every region of phase space. The $\alpha(x)$ bump acts as a trap: each pass through the modified region reduces the orbit's escape velocity until capture.

**What the $\alpha(x)$ approach adds beyond OGY:**
- OGY requires *detecting* when the orbit is near the target UPO and *computing* the correction in real time. The $\alpha(x)$ approach bakes the correction into the map's algebra — it's a **static modification** rather than dynamic feedback.
- OGY perturbs an external *parameter*; $\alpha(x)$ modifies the *iteration rule itself*. This distinction matters: parameter perturbation can only affect the next iterate, while map modification changes the entire phase-space geometry.
- The $\alpha(x)$ formulation makes the stability calculation *exact* (Proposition 1), whereas OGY relies on local linearization.

**Verified computationally:** At $a = 3.8$ (fully chaotic, $\Lambda = +0.44$), a Gaussian $\alpha(x)$ centered on $x^* = 0.737$ with $\alpha(x^*) = 1/(a-1) \approx 0.357$ produces convergence to the fixed point with $\Lambda = -30$ (superstable). This works for all $a \in [3.6, 4.0]$ — the entire chaotic regime.

**References:**
- E. Ott, C. Grebogi, J.A. Yorke, "Controlling chaos," *Phys. Rev. Lett.* 64, 1196–1199 (1990).
- K. Pyragas, "Continuous control of chaos by self-controlling feedback," *Phys. Lett. A* 170, 421–428 (1992). [time-delayed variant]